In [17]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

In [18]:
# Load environment variables from .env file
load_dotenv()

True

In [35]:
# create model Groq

llm = ChatGroq(
    model="qwen/qwen3.8-27b",
    temperature=0.3,
    streaming=True,
)

In [36]:
# Gửi câu hỏi đến Gemini
response = llm.invoke("RAG là gì? Giải thích ngắn gọn bằng tiếng Việt.")

print(response.content)

**RAG** (Retrieval-Augmented Generation - Tạo sinh tăng cường truy xuất) là một kỹ thuật giúp các mô hình ngôn ngữ lớn (LLM) trả lời câu hỏi chính xác hơn bằng cách kết hợp hai bước:

1. **Truy xuất (Retrieval)**: Tìm kiếm thông tin liên quan từ một nguồn dữ liệu bên ngoài (như cơ sở tri thức, tài liệu, web).
2. **Tạo sinh (Generation)**: Sử dụng thông tin vừa tìm được làm ngữ cảnh để mô hình tạo ra câu trả lời.

**Lợi ích chính:**
- Giảm hiện tượng "ảo giác" (hallucination) của AI.
- Cập nhật thông tin mới mà không cần huấn luyện lại mô hình.
- Cho phép AI trả lời dựa trên dữ liệu riêng của tổ chức hoặc người dùng.

Nói ngắn gọn: **RAG giúp AI "tra cứu trước khi trả lời"**, thay vì chỉ dựa vào kiến thức đã học từ trước.


In [37]:
from groq import Groq
client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

for model in client.models.list().data:
    print(model.id)

canopylabs/orpheus-arabic-saudi
meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-safeguard-20b
meta-llama/llama-prompt-guard-2-86m
whisper-large-v3
allam-2-7b
openai/gpt-oss-120b
whisper-large-v3-turbo
canopylabs/orpheus-v1-english
qwen/qwen3.8-27b
openai/gpt-oss-20b


In [38]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

SYSTEM_PROMPT = """
You are a helpful Agentic AI assistant named Tit similar to ChatGPT.

You can:
1. Answer normal questions.
2. Use tools when needed.
3. Search uploaded documents using the RAG tool.
4. Search the web for latest/current information using Tavily Search.
5. Remember important user information using the memory tool.
6. Recall memory when useful.
7. Use calculator for math.

Rules:
- If the user asks about latest news, current events, recent updates, today's information, current prices, current people, current versions, new releases, or anything time-sensitive, use Tavily Search.
- If the user asks about an uploaded document, use search_uploaded_documents.
- If the user asks you to remember something, use remember_this.
- If the user asks about previous preferences or saved facts, use recall_memory.
- Use calculator for math questions.
- When using web search, summarize clearly and mention that the answer is based on web search results.
- Be clear, helpful, and concise.
"""

messages = [
    SystemMessage(content=SYSTEM_PROMPT),
    HumanMessage(content="RAG là gì? Giải thích ngắn gọn bằng tiếng Việt.")
]

response = llm.invoke(messages)
print(response.content)


RAG (Retrieval-Augmented Generation) là một kỹ thuật kết hợp giữa **tìm kiếm thông tin** và **mô hình ngôn ngữ lớn (LLM)**.

Thay vì chỉ dựa vào kiến thức đã học sẵn, RAG giúp AI:
1. **Tìm kiếm** dữ liệu liên quan từ các nguồn bên ngoài (như tài liệu, cơ sở dữ liệu, web).
2. **Sử dụng** dữ liệu đó làm ngữ cảnh để **tạo ra câu trả lời** chính xác và cập nhật hơn.

Nói ngắn gọn: RAG giúp AI "tra cứu" trước khi trả lời, giảm thiểu lỗi "ảo giác" (hallucination) và cung cấp thông tin dựa trên nguồn dữ liệu cụ thể.


In [43]:
from tools import tools
from langchain_core.messages import MessagesState
from langgraph import StateGraph, START, ToolsNode, tools_condition
from langchain_core.messages import SystemMessage
from langchain_core.messages import HumanMessage
from langchain_core.messages import AIMessage
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver



llm_with_tools = llm.with_tools(tools)

# MessagesState là một kiểu trạng thái được LangGraph cung cấp sẵn. Nó định nghĩa cấu trúc state dùng để quản lý danh sách messages.

def chatbot_node(state: MessagesState):
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

workflow = StateGraph(MessagesState) # khai báo rằng workflow sẽ sử dụng MessagesState để quản lý state
workflow.add_node("chatbot", chatbot_node)

toolNode = ToolsNode(tools)
workflow.add_node("tools", toolNode)
workflow.add_edge(START, "chatbot")
workflow.add_edge("chatbot", "tools")
workflow.add_conditional_edges("chatbot", tools_condition) # mac dinh tim node ten la tools

conn = sqlite3.connect(
    "data/langgraph_checkpoints.sqlite",
    check_same_thread=False
)

checkpoint = SqliteSaver(conn)
workflow.compile(checkpoint=checkpoint)
workflow



ModuleNotFoundError: No module named 'database'

In [44]:
import os

print(os.getcwd())
print(os.listdir())

d:\Tit\experiments
['1-model_create.ipynb']
